# NPS Preditivo: Análise Exploratória de Dados

**Tech Challenge Fase 1 - FIAP Pós Tech AI Scientist**

**Objetivo:** Identificar quais fatores operacionais impactam a satisfação do cliente (NPS) em um e-commerce e apontar como esses dados podem apoiar ações proativas antes da pesquisa de NPS ser aplicada.

**Base:** `desafio_nps_fase_1.csv` com 2.500 registros de pedidos contendo dados de logística, atendimento e a nota NPS dada pelo cliente.

**Metodologia:** CRISP-DM aplicado às etapas de entendimento dos dados e análise exploratória.

## 1. Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

## 2. Leitura e Visão Geral dos Dados

In [ ]:
df = pd.read_csv('../data/raw/desafio_nps_fase_1.csv')

print(f'Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas')
print(f'Valores faltantes: {df.isnull().sum().sum()} (base limpa)')
df.head()

In [ ]:
df.describe().T.round(2)

## 3. Análise da Variável Target: nps_score

O `nps_score` é a nota que o cliente dá ao final da compra, de 0 a 10. Seguindo a metodologia NPS padrão:
- **Promotores:** nota 9 ou 10
- **Neutros:** nota 7 ou 8
- **Detratores:** nota 0 a 6

**NPS final = % Promotores - % Detratores**

In [ ]:
def categorizar_nps(nota):
    if nota >= 9:
        return 'Promotor'
    elif nota >= 7:
        return 'Neutro'
    else:
        return 'Detrator'

df['nps_categoria'] = df['nps_score'].apply(categorizar_nps)

contagem = df['nps_categoria'].value_counts()
percentual = contagem / len(df) * 100

for cat in ['Promotor', 'Neutro', 'Detrator']:
    print(f'{cat}: {contagem[cat]} ({percentual[cat]:.1f}%)')

pct_p = percentual.get('Promotor', 0)
pct_d = percentual.get('Detrator', 0)
print(f'\nNPS agregado da empresa: {pct_p - pct_d:.1f}')
print(f'Média nps_score: {df["nps_score"].mean():.2f}')
print(f'Mediana nps_score: {df["nps_score"].median():.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cores = {'Promotor': '#2ecc71', 'Neutro': '#f39c12', 'Detrator': '#e74c3c'}
ordem = ['Promotor', 'Neutro', 'Detrator']
vals = [contagem[c] for c in ordem]

axes[0].bar(ordem, vals, color=[cores[c] for c in ordem], edgecolor='white')
axes[0].set_title('Clientes por categoria NPS')
axes[0].set_ylabel('Quantidade')
for idx, (cat, val) in enumerate(zip(ordem, vals)):
    axes[0].text(idx, val + 15, f'{val}\n({percentual[cat]:.1f}%)', ha='center', fontsize=9)

axes[1].hist(df['nps_score'], bins=30, color='#3498db', edgecolor='white', linewidth=0.5)
axes[1].axvline(df['nps_score'].mean(), color='red', linestyle='--',
                label=f'Media: {df["nps_score"].mean():.1f}')
axes[1].axvline(df['nps_score'].median(), color='orange', linestyle='--',
                label=f'Mediana: {df["nps_score"].median():.1f}')
axes[1].set_title('Distribuicao do nps_score')
axes[1].set_xlabel('nps_score')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/fig_01_distribuicao_nps.png', bbox_inches='tight', dpi=120)
plt.show()

## 4. Correlação com o NPS

Calculamos a correlação de Spearman entre cada variável numérica e o `nps_score`. Spearman é preferível aqui por não exigir relações lineares.

In [ ]:
variaveis_num = [
    'customer_age', 'customer_tenure_months', 'order_value', 'items_quantity',
    'discount_value', 'payment_installments', 'delivery_time_days',
    'delivery_delay_days', 'freight_value', 'delivery_attempts',
    'customer_service_contacts', 'resolution_time_days',
    'complaints_count', 'repeat_purchase_30d', 'csat_internal_score'
]

corrs = (
    df[variaveis_num + ['nps_score']]
    .corr(method='spearman')['nps_score']
    .drop('nps_score')
    .sort_values()
)

for var, val in corrs.items():
    print(f'{var:35s}: {val:+.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cores_barras = ['#e74c3c' if c < 0 else '#2ecc71' for c in corrs.values]
ax.barh(corrs.index, corrs.values, color=cores_barras, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlacao de Spearman com nps_score')
ax.set_title('Correlacao das variaveis operacionais com o NPS')
for idx, (var, val) in enumerate(corrs.items()):
    ax.text(val + (0.01 if val >= 0 else -0.01), idx, f'{val:+.2f}',
            va='center', ha='left' if val >= 0 else 'right', fontsize=8)
plt.tight_layout()
plt.savefig('../reports/fig_02_correlacoes_nps.png', bbox_inches='tight', dpi=120)
plt.show()

**Principais achados da correlação:**

- `delivery_delay_days` (-0,59): atraso na entrega é o fator mais impactante negativamente.
- `csat_internal_score` (+0,56): score interno alinhado com o NPS.
- `complaints_count` (-0,49): mais reclamações, menos satisfação.
- `repeat_purchase_30d` (+0,49): clientes satisfeitos voltam a comprar.
- `customer_service_contacts` (-0,34): mais contatos com atendimento indicam problemas.
- `delivery_time_days` (0,00): tempo total de entrega não tem correlação. O que importa é o atraso em relação ao prazo prometido.

## 5. Análise Bivariada: Fatores Chave vs. NPS

In [ ]:
variaveis_chave = ['delivery_delay_days', 'csat_internal_score',
                   'complaints_count', 'customer_service_contacts']
titulos = ['Dias de atraso', 'Score CSAT interno', 'Reclamacoes', 'Contatos atendimento']
ordem_cat = ['Promotor', 'Neutro', 'Detrator']

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()
cores_box = {'Promotor': '#2ecc71', 'Neutro': '#f39c12', 'Detrator': '#e74c3c'}

for i, (var, titulo) in enumerate(zip(variaveis_chave, titulos)):
    dados = [df[df['nps_categoria'] == cat][var].values for cat in ordem_cat]
    bp = axes[i].boxplot(dados, labels=ordem_cat, patch_artist=True,
                         medianprops={'color': 'black', 'linewidth': 2})
    for patch, cat in zip(bp['boxes'], ordem_cat):
        patch.set_facecolor(cores_box[cat])
        patch.set_alpha(0.7)
    medias = [df[df['nps_categoria'] == cat][var].mean() for cat in ordem_cat]
    axes[i].scatter(range(1, 4), medias, color='navy', zorder=5, s=40, label='Media')
    axes[i].set_title(titulo)
    axes[i].legend(fontsize=8)

plt.suptitle('Distribuicao dos fatores por categoria NPS', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../reports/fig_03_boxplots_fatores_nps.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
print('Medias por categoria NPS:')
print(df.groupby('nps_categoria')[variaveis_chave + ['resolution_time_days']].mean().round(2).loc[ordem_cat].to_string())

## 6. Ponto de Ruptura: Atraso na Entrega

In [ ]:
def faixa_atraso(d):
    if d == 0: return '0 (no prazo)'
    elif d <= 2: return '1-2 dias'
    elif d <= 5: return '3-5 dias'
    else: return '6+ dias'

df['faixa_atraso'] = df['delivery_delay_days'].apply(faixa_atraso)

ordem_faixa = ['0 (no prazo)', '1-2 dias', '3-5 dias', '6+ dias']
resumo_faixa = df.groupby('faixa_atraso', sort=False).agg(
    n_clientes=('nps_score', 'count'),
    nps_medio=('nps_score', 'mean')
).round(2).reindex(ordem_faixa)

print('NPS medio por faixa de atraso:')
print(resumo_faixa.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

nps_por_atraso = df.groupby('delivery_delay_days')['nps_score'].mean().reset_index()
axes[0].plot(nps_por_atraso['delivery_delay_days'], nps_por_atraso['nps_score'],
             marker='o', color='#e74c3c', linewidth=2, markersize=5)
axes[0].axhline(7, color='green', linestyle='--', alpha=0.5, label='Minimo neutro (7)')
axes[0].set_xlabel('Dias de atraso')
axes[0].set_ylabel('NPS medio')
axes[0].set_title('NPS medio por dias de atraso')
axes[0].legend()

cores_faixa = ['#2ecc71', '#f39c12', '#e57e22', '#e74c3c']
nps_faixa = [resumo_faixa.loc[f, 'nps_medio'] for f in ordem_faixa]
n_faixa = [resumo_faixa.loc[f, 'n_clientes'] for f in ordem_faixa]

bars = axes[1].bar(ordem_faixa, nps_faixa, color=cores_faixa, edgecolor='white')
axes[1].set_ylabel('NPS medio')
axes[1].set_title('NPS medio por faixa de atraso')
axes[1].set_ylim(0, 9)
for bar, val, n in zip(bars, nps_faixa, n_faixa):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.1,
                 f'{val:.2f}\n(n={int(n)})', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('../reports/fig_04_ruptura_atraso.png', bbox_inches='tight', dpi=120)
plt.show()
print('Ponto de ruptura: a partir de 3 dias de atraso, NPS cai de 5.05 para 2.89.')

## 7. Impacto dos Contatos com Atendimento

In [ ]:
nps_contato = df.groupby('customer_service_contacts').agg(
    n=('nps_score','count'), nps_medio=('nps_score','mean')).round(2)

print('NPS medio por numero de contatos:')
print(nps_contato.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(nps_contato.index, nps_contato['nps_medio'], color='#3498db', edgecolor='white')
ax.set_xlabel('Numero de contatos com atendimento')
ax.set_ylabel('NPS medio')
ax.set_title('NPS medio por numero de contatos com atendimento')
for _, row in nps_contato.iterrows():
    ax.text(row.name, row['nps_medio'] + 0.1,
            f'{row["nps_medio"]:.2f}\n(n={int(row["n"])})', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('../reports/fig_05_atendimento_nps.png', bbox_inches='tight', dpi=120)
plt.show()

## 8. Recompra em 30 Dias por Categoria NPS

In [ ]:
recompra = df.groupby('nps_categoria')['repeat_purchase_30d'].mean() * 100
recompra = recompra.loc[['Promotor', 'Neutro', 'Detrator']]

print('Taxa de recompra em 30 dias:')
for cat, val in recompra.items():
    print(f'  {cat}: {val:.1f}%')

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(recompra.index, recompra.values,
              color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='white')
ax.set_ylabel('% que recomprou em 30 dias')
ax.set_title('Taxa de recompra por categoria NPS')
ax.set_ylim(0, 115)
for bar, val in zip(bars, recompra.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 2,
            f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/fig_06_recompra_por_nps.png', bbox_inches='tight', dpi=120)
plt.show()

## 9. Analise por Regiao

In [ ]:
nps_regiao = df.groupby('customer_region').agg(
    n=('nps_score','count'),
    nps_medio=('nps_score','mean'),
    pct_detrator=('nps_categoria', lambda x: (x=='Detrator').mean()*100)
).round(2).sort_values('nps_medio')

print('NPS por regiao:')
print(nps_regiao.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(nps_regiao.index, nps_regiao['nps_medio'], color='#5dade2', edgecolor='white')
ax.set_ylabel('NPS medio')
ax.set_title('NPS medio por regiao')
ax.set_ylim(0, 7)
for i, (reg, val) in enumerate(nps_regiao['nps_medio'].items()):
    ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../reports/fig_07_nps_por_regiao.png', bbox_inches='tight', dpi=120)
plt.show()
print('Diferenca entre regioes e pequena (4.21 a 4.49). O problema nao e geografico.')

## 10. Perfil: Promotores vs. Detratores

In [ ]:
cols_perfil = ['delivery_delay_days', 'customer_service_contacts',
               'complaints_count', 'resolution_time_days', 'csat_internal_score']
labels_perfil = ['Atraso entrega', 'Contatos atendimento', 'Reclamacoes',
                 'Tempo resolucao', 'CSAT interno']

promo_vals = [df[df['nps_categoria']=='Promotor'][v].mean() for v in cols_perfil]
detrat_vals = [df[df['nps_categoria']=='Detrator'][v].mean() for v in cols_perfil]

x = range(len(labels_perfil))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar([i - width/2 for i in x], promo_vals, width, label='Promotor', color='#2ecc71', alpha=0.85)
ax.bar([i + width/2 for i in x], detrat_vals, width, label='Detrator', color='#e74c3c', alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(labels_perfil)
ax.set_ylabel('Valor medio')
ax.set_title('Indicadores operacionais: Promotores vs. Detratores')
ax.legend()
for i, (p, d) in enumerate(zip(promo_vals, detrat_vals)):
    ax.text(i - width/2, p + 0.05, f'{p:.2f}', ha='center', fontsize=8, color='darkgreen')
    ax.text(i + width/2, d + 0.05, f'{d:.2f}', ha='center', fontsize=8, color='darkred')
plt.tight_layout()
plt.savefig('../reports/fig_08_perfil_promotor_detrator.png', bbox_inches='tight', dpi=120)
plt.show()

## 11. Preparacao da Base Tratada

In [ ]:
df.to_csv('../data/processed/nps_tratado.csv', index=False)
print(f'Base tratada salva: {df.shape[0]} linhas x {df.shape[1]} colunas')
print('Colunas adicionadas: nps_categoria, faixa_atraso')
print('Arquivo: data/processed/nps_tratado.csv')

## 12. Sintese: O que os Dados Dizem para o Negocio

### Situacao atual

O NPS da empresa nesta amostra e de **-69,6**. De cada 100 clientes, 84 sao detratores, 11 sao neutros e apenas 4 sao promotores.

### Fatores mais criticos para a satisfacao

**1. Atraso na entrega** e o principal problema (correlacao -0,59):
- Entregas no prazo: NPS medio 6,86
- 1-2 dias de atraso: NPS medio 5,05
- 3-5 dias de atraso: NPS medio 2,89
- 6+ dias de atraso: NPS medio 0,81

**2. Reclamacoes** (-0,49): detratores acumulam em media 4,4 reclamacoes, contra 2,3 dos promotores.

**3. Contatos com atendimento** (-0,34): cada contato adicional reduz o NPS. Sem contato: media 5,54. Com 5 contatos: media 2,24.

### O que NAO importa tanto quanto parece

O tempo total de entrega nao tem correlacao com o NPS (0,00). A regiao geografica e o tempo de relacionamento com a empresa tambem nao discriminam a satisfacao de forma relevante.

### Impacto direto na receita

Promotores: 100% voltam a comprar em 30 dias. Detratores: 0% voltam. Isso conecta NPS diretamente a receita futura.

### Proximo passo

Com esses padroes, e viavel construir um modelo que preveja o NPS de um cliente com base nos dados operacionais durante a jornada, antes da pesquisa ser enviada.